# Notebook 1 - Data Pipeline
CSE 3104 Final Project | Andrew, Imaan, Juliet | April 2026

---

this notebook builds the main table we use for the regression analysis. it loads census income and population data, counts store locations per zip code for 4 chains, merges racial demographic data, and calculates density per 10,000 residents.

run this one first.

inputs (all in data/ folder):
- ACSDT5Y2024.B19013-Data.csv
- ACSDP5Y2024.DP05-Data.csv
- mcdonalds_locations.csv
- burger_king_locations.csv
- wendys_restaurants.csv
- carlsjr_locations.csv
- demographics.csv

outputs:
- outputs/pipeline_output.csv (full national table, all zip codes)
- outputs/five_city_analysis.csv (filtered to Cleveland, St. Louis, Santa Clara, San Francisco, Seattle)


In [42]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR   = 'data'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def data(f):   return os.path.join(DATA_DIR, f)
def output(f): return os.path.join(OUTPUT_DIR, f)

## Load Census Data

loads income and population from the two ACS csv files. drops the label row census includes by default, renames columns, and converts to numeric.

In [43]:
census = pd.read_csv(data('ACSDT5Y2024.B19013-Data.csv'), skiprows=[1])
census['zip_code']      = census['NAME'].str.extract(r'(\d{5})')
census['median_income'] = pd.to_numeric(census['B19013_001E'], errors='coerce')
census = census[['zip_code', 'median_income']].dropna(subset=['zip_code'])

# impute missing income using state-prefix (first 2 digits of ZIP) average
census['state'] = census['zip_code'].str[:2]
missing_before  = census['median_income'].isna().sum()
census['median_income'] = census.groupby('state')['median_income'].transform(
    lambda x: x.fillna(x.mean())
)
census = census.drop(columns='state')

print(f'Loaded {len(census):,} ZIPs  |  imputed {missing_before} missing income values')
census.head(3)

Loaded 33,772 ZIPs  |  imputed 3358 missing income values


,zip_code,median_income
0,00601,19454.0
1,00602,21420.0
2,00603,20933.0


## Load Restaurant Locations

reads each chain's csv and counts locations per zip code.

In [44]:
pop_raw = pd.read_csv(data('ACSDP5Y2024.DP05-Data.csv'), skiprows=[1], low_memory=False)
pop = pop_raw[['NAME', 'DP05_0001E']].copy()
pop['zip_code']         = pop['NAME'].str.extract(r'(\d{5})')
pop['total_population'] = pd.to_numeric(pop['DP05_0001E'], errors='coerce')
pop = pop[['zip_code', 'total_population']].dropna(subset=['zip_code'])

print(f'Loaded {len(pop):,} ZIPs with population data')
pop.head(3)

Loaded 33,772 ZIPs with population data


,zip_code,total_population
0,00601,16669
1,00602,37233
2,00603,48448


## Merge Everything

merges census income, population, and all chain counts into one table. zips with no stores get 0, not dropped.

In [45]:
def count_per_zip(filepath, zip_col, count_col, filter_col=None, filter_val=None):
    """Load a restaurant CSV and return a ZIP -> store count table.
    filter_col/filter_val optionally keep only rows where filter_col == filter_val (used to drop international locations)."""
    df = pd.read_csv(filepath)
    if filter_col and filter_val:
        df = df[df[filter_col] == filter_val]
    df = df.dropna(subset=[zip_col])
    # split on '.' before zfill to handle float-formatted zips like 10001.0
    df['zip_code'] = df[zip_col].astype(str).str.split('.').str[0].str.zfill(5).str[:5]
    counts = df.groupby('zip_code').size().reset_index(name=count_col)
    print(f'  {count_col:<25}  {len(df):>6,} locations  ->  {len(counts):,} ZIPs')
    return counts

print('Restaurant location files:')
chains = [
    count_per_zip(data('mcdonalds_locations.csv'),   'zipcode',   'mcdonalds_count'),
    count_per_zip(data('burger_king_locations.csv'), 'zipcode',   'burger_king_count'),
    count_per_zip(data('wendys_restaurants.csv'),    'zipcode',   'wendys_count',    filter_col='country', filter_val='US'),
    count_per_zip(data('carlsjr_locations.csv'),     'zip_code',  'carlsjr_count'),
]


Restaurant location files:
  mcdonalds_count            13,418 locations  ->  9,175 ZIPs
  burger_king_count           6,642 locations  ->  5,241 ZIPs
  wendys_count                5,736 locations  ->  4,800 ZIPs
  carlsjr_count               2,789 locations  ->  2,387 ZIPs


## Calculate Density

converts raw counts to stores per 10,000 people so zips of different sizes are comparable.

In [46]:
# merge census + population + all 4 chains
master = census.merge(pop, on='zip_code', how='left')
for chain_df in chains:
    master = master.merge(chain_df, on='zip_code', how='left')

count_cols = ['mcdonalds_count', 'burger_king_count', 'wendys_count', 'carlsjr_count']
master[count_cols] = master[count_cols].fillna(0).astype(int)

# tier 1 = all four value chains
master['total_tier1_count'] = master[count_cols].sum(axis=1)

# density per 10k residents (NaN where population is 0)
valid_pop = master['total_population'] > 0
for col in count_cols + ['total_tier1_count']:
    dcol = col.replace('_count', '_density')
    master[dcol] = np.where(valid_pop,
                            (master[col] / master['total_population']) * 10000,
                            np.nan)

print(f'Master table: {master.shape[0]:,} rows  x  {master.shape[1]} columns')
print('Store totals:')
for col in count_cols + ['total_tier1_count']:
    print(f'  {col:<25}  {master[col].sum():>6,}')
master.head(3)


Master table: 33,772 rows  x  13 columns
Store totals:
  mcdonalds_count            13,312
  burger_king_count           6,612
  wendys_count                5,726
  carlsjr_count               2,777
  total_tier1_count          28,427


,zip_code,median_income,total_population,mcdonalds_count,burger_king_count,wendys_count,carlsjr_count,total_tier1_count,mcdonalds_density,burger_king_density,wendys_density,carlsjr_density,total_tier1_density
0,00601,19454.0,16669,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
1,00602,21420.0,37233,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0
2,00603,20933.0,48448,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0


In [47]:
# load acs racial breakdown (b02001) — keep only estimate columns, drop margins of error
demo_raw = pd.read_csv(data('demographics.csv'))

keep_cols = [
    'ZIP',
    'Estimate!!Total:',
    'Estimate!!Total:!!White alone',
    'Estimate!!Total:!!Black or African American alone',
    'Estimate!!Total:!!Asian alone',
    'Estimate!!Total:!!Some Other Race alone',
]
demo = demo_raw[keep_cols].copy()
demo.columns = ['zip_code', 'total_pop_race', 'white_count', 'black_count', 'asian_count', 'other_count']

# zip is stored as an integer in this file so cast to zero-padded string
demo['zip_code'] = demo['zip_code'].astype(str).str.zfill(5).str[:5]

for col in ['total_pop_race', 'white_count', 'black_count', 'asian_count', 'other_count']:
    demo[col] = pd.to_numeric(demo[col], errors='coerce')

# calculate percentages — avoid division by zero
valid_race_pop = demo['total_pop_race'] > 0
for race, pct in [('white_count', 'pct_white'), ('black_count', 'pct_black'),
                  ('asian_count', 'pct_asian'), ('other_count', 'pct_other')]:
    demo[pct] = np.where(valid_race_pop,
                         (demo[race] / demo['total_pop_race'] * 100).round(2),
                         np.nan)

demo = demo[['zip_code', 'pct_white', 'pct_black', 'pct_asian', 'pct_other']]

# left join keeps all master zips; zips outside demo coverage get NaN
master = master.merge(demo, on='zip_code', how='left')

matched = master['pct_white'].notna().sum()
print(f'zip codes with demographic data: {matched:,} of {len(master):,}')


zip codes with demographic data: 4,607 of 33,772


In [48]:
# filter to target cities using zip prefix — each prefix maps to one metro area
city_prefixes = {
    '441': 'Cleveland',
    '631': 'St. Louis',
    '950': 'Santa Clara',
    '941': 'San Francisco',
    '981': 'Seattle',
}

prefix_pattern = '|'.join(f'^{p}' for p in city_prefixes.keys())
five_city = master[master['zip_code'].str.match(prefix_pattern)].copy()

# label each row with the city name for easier grouping later
five_city['city'] = five_city['zip_code'].apply(
    lambda z: next((name for prefix, name in city_prefixes.items() if z.startswith(prefix)), None)
)

# recalculate tier1 using only the four target chains for this export
five_city['total_tier1_count'] = (
    five_city['mcdonalds_count'] + five_city['burger_king_count']
    + five_city['wendys_count'] + five_city['carlsjr_count']
)

# recalculate tier1 density — NaN where population is zero
valid_pop = five_city['total_population'] > 0
five_city['total_tier1_density'] = np.where(
    valid_pop,
    (five_city['total_tier1_count'] / five_city['total_population']) * 10000,
    np.nan
)

# keep only the four-chain columns for the five-city export
five_city_cols = [
    'zip_code', 'city', 'median_income', 'total_population',
    'mcdonalds_count', 'burger_king_count', 'wendys_count', 'carlsjr_count', 'total_tier1_count',
    'mcdonalds_density', 'burger_king_density', 'wendys_density', 'carlsjr_density', 'total_tier1_density',
    'pct_white', 'pct_black', 'pct_asian', 'pct_other',
]
five_city = five_city[five_city_cols]

five_city.to_csv(output('five_city_analysis.csv'), index=False)
print(f'five-city table: {len(five_city):,} zip codes  x  {len(five_city.columns)} columns  saved to outputs/five_city_analysis.csv')
print(five_city['city'].value_counts().to_string())


five-city table: 201 zip codes  x  18 columns  saved to outputs/five_city_analysis.csv
city
Cleveland        48
St. Louis        46
Santa Clara      39
Seattle          39
San Francisco    29


## Export

saves the final table to outputs/pipeline_output.csv.

In [49]:
master.to_csv(output('pipeline_output.csv'), index=False)
print(f'Saved  outputs/pipeline_output.csv  ({len(master):,} rows  ×  {len(master.columns)} columns)')

Saved  outputs/pipeline_output.csv  (33,772 rows  ×  17 columns)
